# 03 - Test the Claude Code compatible endpoint

This sends an Anthropic Messages API request to LiteLLM. LiteLLM translates it to the Foundry GPT-5.4 deployment. The same URL and token are the values Claude Code consumes.

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv

candidate_roots = [Path.cwd(), *Path.cwd().parents]
demo_root = next((root / 'demos' / 'litellm01-foundry-claude' for root in candidate_roots if (root / 'demos' / 'litellm01-foundry-claude').is_dir()), None)
if demo_root is None:
    raise RuntimeError('Could not locate demos/litellm01-foundry-claude from the notebook working directory.')
env_path = demo_root / 'env' / '.env'
load_dotenv(env_path)
proxy_url = os.environ['LITELLM_PROXY_URL'].rstrip('/')
proxy_key = (os.getenv('ANTHROPIC_AUTH_TOKEN', '').strip() or os.getenv('LITELLM_CLIENT_KEY', '').strip() or os.environ['LITELLM_MASTER_KEY'])
model_alias = os.environ['LITELLM_MODEL_ALIAS']

print('Claude Code configuration:')
print(f'$env:ANTHROPIC_BASE_URL = {proxy_url}')
print(f'$env:ANTHROPIC_AUTH_TOKEN = {proxy_key[:8]}...')
print(f"claude --model {model_alias}")

## Step 1 - Send an Anthropic Messages API request

In [ ]:
import urllib.error
import urllib.request

payload = {
    'model': model_alias,
    'max_tokens': 120,
    'messages': [{'role': 'user', 'content': 'In one sentence, explain why a public model gateway should use an API key.'}],
}
request = urllib.request.Request(
    url=f'{proxy_url}/v1/messages',
    data=json.dumps(payload).encode('utf-8'),
    headers={'Authorization': f'Bearer {proxy_key}', 'Content-Type': 'application/json'},
    method='POST',
)
try:
    with urllib.request.urlopen(request, timeout=180) as response:
        status_code = response.getcode()
        body_text = response.read().decode('utf-8')
except urllib.error.HTTPError as exc:
    status_code = exc.code
    body_text = exc.read().decode('utf-8', errors='replace')
if not (200 <= status_code < 300):
    raise RuntimeError(f'Messages API request failed ({status_code}): {body_text[:500]}')
result = json.loads(body_text)
Path('../outputs').mkdir(parents=True, exist_ok=True)
Path('../outputs/last-message.json').write_text(json.dumps(result, indent=2), encoding='utf-8')
print(result.get('content', result))
print('Anthropic Messages API compatibility: OK')